# Plot Protein LM Training Metrics

Read `training_metrics.jsonl` and optional `resume_training_metrics.jsonl` from the protein checkpoint directory, then plot train and validation loss against optimizer steps.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
METRIC_FILES = [
    CHECKPOINT_DIR / "training_metrics.jsonl",
    CHECKPOINT_DIR / "resume_training_metrics.jsonl",
]
PLOT_PATH = CHECKPOINT_DIR / "loss_curve.png"

PROJECT_ROOT

In [ ]:
rows = []
for metric_file in METRIC_FILES:
    if not metric_file.exists():
        continue
    for line in metric_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            row["metric_file"] = str(metric_file)
            rows.append(row)

if not rows:
    raise FileNotFoundError(
        f"No metric JSONL files found under {CHECKPOINT_DIR}. Run notebook 03 or 04 first."
    )

rows = sorted(rows, key=lambda item: int(item.get("global_step", 0)))
rows[:3], rows[-3:]

In [ ]:
steps = [int(row["global_step"]) for row in rows]
train_losses = [float(row["train_loss"]) for row in rows]
val_losses = [float(row["val_loss"]) for row in rows]
tokens_seen = [int(row.get("tokens_seen", 0)) for row in rows]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(steps, train_losses, label="Train loss", linewidth=2)
ax1.plot(steps, val_losses, label="Validation loss", linewidth=2)
ax1.set_title("Protein LM Training Metrics")
ax1.set_xlabel("Optimizer step")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.25)
ax1.legend()

ax2 = ax1.twiny()
ax2.plot(tokens_seen, train_losses, alpha=0)
ax2.set_xlabel("Tokens seen")

fig.tight_layout()
PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_PATH, dpi=150)
PLOT_PATH